# ML Platform Research: Exploratory Data Analysis

## 1. Introduction

The objective of this notebook is to explore the cleaned RetailRocket datasets in order to understand user behavior, product interactions, and temporal purchasing patterns.

Unlike the previous Data Assessment notebook, which focused on data quality and cleaning decisions, this stage aims to extract business insights and identify opportunities for feature engineering. Every observation made during the exploratory analysis should contribute to answering the following questions:

- Which historical behaviors are associated with purchasing activity?
- Which product characteristics may provide useful predictive information?
- Which temporal patterns should be represented?
- Which features should be prioritized for the initial modeling approach?
- How can predictive information be captured while preventing target leakage?

The analyses performed in this notebook will guide the design of the Gold layer and define the feature engineering requirements for the production pipeline.

## 2. Imports and Dataset Load

In [ ]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("..")

from utils import (
    plot_bar, 
    plot_histogram, 
    plot_scatter, 
    plot_line, 
    pareto_analysis,
    build_funnel,
    compute_conversion_rates,
    conversion_summary,
    build_category_depth
)

events = pd.concat(
    [
        pd.read_parquet(file)
        for file in sorted(Path("../data/clean/events").glob("*.parquet"))
    ],
    ignore_index=True,
)

In [ ]:
item_properties = pd.read_parquet("../data/clean/item_properties.parquet")

In [ ]:
category_tree = pd.read_parquet("../data/clean/category_tree.parquet")

## 3. Events

The events dataset records user interactions with products and constitutes the primary source for the supervised learning problem.

Each record represents a single event generated by a visitor at a specific moment in time.

### 3.1 Overview

This section provides a general overview of the cleaned events dataset before exploring specific behavioral patterns.

In [ ]:
print(f"Memory usage: {events.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

events

In [ ]:
events.info()

In [ ]:
events.describe(include="all")

### 3.2 Event Distribution

In [ ]:
events["event"].value_counts()

In [ ]:
events["event"].value_counts(normalize=True).mul(100)

User interactions are highly dominated by product views, while purchases represent only a small fraction of all recorded events.

### 3.3 Visitor Behavior

#### 3.3.1 Activity Distribution

In [ ]:
visitor_events = (
    events
    .groupby("visitorid")["event"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

visitor_events.sort_values(by="transaction", ascending=False).head(20)

In [ ]:
plot_histogram(visitor_events, "view", 100, "Views x Visitor", "Number of Views", "Number of Visitors")

In [ ]:
visitor_events["view"].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999])

In [ ]:
plot_histogram(visitor_events, "addtocart", 100, "Add to cart x Visitor", "Number of Adds", "Number of Visitors")

In [ ]:
visitor_events["addtocart"].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999])

In [ ]:
plot_histogram(visitor_events, "transaction", 100, "Transactions x Visitor", "Number of Transactions", "Number of Visitors")

In [ ]:
visitor_events["transaction"].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999])

The distributions are highly right-skewed, indicating that most visitors interact with only a few products while a small fraction is extremely active.

#### 3.3.2 Relationship Between Event Types

In [ ]:
plot_scatter(df=visitor_events, x="view", y="transaction", hue=None, title="View vs Transaction", xlabel="Nº views", ylabel="Nº transactions")

In [ ]:
view_to_transaction_ratio = (
    visitor_events["transaction"].sum()
    / visitor_events["view"].sum()
)*100

view_to_transaction_ratio

In [ ]:
plot_scatter(df=visitor_events, x="view", y="addtocart", hue=None, title="Views vs Add to Cart", xlabel="Nº views", ylabel="Nº add to cart")

In [ ]:
(visitor_events["addtocart"].sum()/visitor_events["view"].sum())*100

view_to_cart_ratio = (
    visitor_events["addtocart"].sum()
    / visitor_events["view"].sum()
)*100

view_to_cart_ratio

In [ ]:
plot_scatter(df=visitor_events, x="addtocart", y="transaction", hue=None, title="Add to Cart vs Transaction", xlabel="Nº add to cart", ylabel="Nº transactions")

In [ ]:
cart_to_transaction_ratio = (
    visitor_events["transaction"].sum()
    / visitor_events["addtocart"].sum()
)*100

cart_to_transaction_ratio

The scatter plots reveal a strong concentration of users with little or no purchasing activity, together with a small group of highly engaged users responsible for a disproportionate number of transactions.

#### 3.3.3 Pareto Analysis

In [ ]:
pareto = pareto_analysis(
    df=visitor_events,
    value_col="transaction",
    entity_name="visitors",
    threshold=0.80
)

In [ ]:
plot_line(
    df=pareto,
    x="cum_visitors_pct",
    y="cum_transaction_pct",
    title="Pareto Analysis of Transactions",
    xlabel="Cumulative Percentage of Visitors",
    ylabel="Cumulative Percentage of Transactions",
    diagonal=True
)

A small fraction of visitors is responsible for most purchases, suggesting that historical customer activity may provide useful predictive signals.

#### 3.3.4 Top Visitors

To visualize how concentrated the number of transactions are in some customers, we plot the top 30 customers by number of transactions.

In [ ]:
top_visitors = (
    visitor_events
    .sort_values("transaction", ascending=False)
    .head(30)
    .reset_index()
)

top_visitors

In [ ]:
plot_bar(
    df=top_visitors,
    x="visitorid",
    y="transaction",
    title="Top 30 Visitors by Number of Transactions",
    xlabel="Visitor ID",
    ylabel="Transactions",
    rotation=90,
    order=top_visitors["visitorid"]
)

### 3.4 Item Behavior

Products also exhibit highly heterogeneous purchasing frequencies. Understanding this concentration helps identify popularity effects that may be useful during feature engineering.

#### 3.4.1 Pareto Analysis

In [ ]:
item_events = (
    events
    .groupby("itemid")["event"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

if "transaction" not in item_events.columns:
    item_events["transaction"] = 0

item_events.sort_values(by="transaction", ascending=False).head(20)

In [ ]:
pareto = pareto_analysis(
    df=item_events,
    value_col="transaction",
    entity_name="items",
    threshold=0.80
)

In [ ]:
plot_line(
    df=pareto,
    x="cum_items_pct",
    y="cum_transaction_pct",
    title="Pareto Analysis of Transactions by Item",
    xlabel="Cumulative Percentage of Items",
    ylabel="Cumulative Percentage of Transactions",
    diagonal=True
)

Although less acute compared to the concentration of purchases among visitors, a small fraction of items is responsible for most transactions. This indicates that item popularity may provide useful predictive information and motivates the exploration of aggregate popularity-based features rather than relying directly on item identifiers.

#### 3.4.2 Top Items

As with the top customers, we visualize the top 30 items by number of transactions.

In [ ]:
top_items = (
    item_events
    .sort_values("transaction", ascending=False)
    .head(30)
    .reset_index()
)

top_items

In [ ]:
plot_bar(
    df=top_items,
    x="itemid",
    y="transaction",
    title="Top 30 Items by Number of Transactions",
    xlabel="Item ID",
    ylabel="Transactions",
    rotation=90,
    order=top_items["itemid"]
)

### 3.5 Temporal Analysis

#### 3.5.1 Monthly Activity

In [ ]:
monthly_events = (
    events
    .groupby(
        [
            events["timestamp"].dt.to_period("M"),
            "event"
        ]
    )
    .size()
    .reset_index(name="count")
)

monthly_events["timestamp"] = monthly_events["timestamp"].astype(str)

monthly_events

In [ ]:
plot_bar(
    df=monthly_events,
    x="timestamp",
    y="count",
    hue="event",
    title="Monthly Event Volume",
    xlabel="Month",
    ylabel="Number of Events"
)

The volume of views is too overwhelming, impeding the proper visualization of `addtocart` and `transaction`. We then remove the `view` event from the next chart.

In [ ]:
plot_bar(
    df=monthly_events[monthly_events["event"] != "view"],
    x="timestamp",
    y="count",
    hue="event",
    title="Monthly Event Volume",
    xlabel="Month",
    ylabel="Number of Events"
)

#### 3.5.2 Daily Activity

In [ ]:
daily_transactions = (
    events.loc[events["event"] == "transaction"]
    .groupby(events["timestamp"].dt.floor("D"))
    .size()
    .rename("transactions")
    .reset_index()
)

In [ ]:
plot_line(
    df=daily_transactions,
    x="timestamp",
    y="transactions",
    title="Daily Activity",
    xlabel="Date",
    ylabel="Number of Transactions"
)

It is possible to see a wave-like pattern. We zoom in to better visualize it.

In [ ]:
plot_line(
    df=daily_transactions[daily_transactions["timestamp"] < "2015-06-15"],
    x="timestamp",
    y="transactions",
    title="Daily Activity",
    xlabel="Date",
    ylabel="Number of Transactions"
)

It is possible to notice a spike approximately in the middle of the week. We deeply inspect the transactions in each day of the week.

#### 3.5.3 Weekly Patterns

In [ ]:
weekday_events = (
    events
    .groupby(
        [
            events["timestamp"].dt.day_name(),
            "event"
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

weekday_events

In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_events["timestamp"] = pd.Categorical(
    weekday_events["timestamp"],
    categories=weekday_order,
    ordered=True
)

weekday_events = weekday_events.sort_values("timestamp")

In [ ]:
plot_line(
    df=weekday_events,
    x="timestamp",
    y="transaction",
    title="Weekly Activity",
    xlabel="Day of the Week",
    ylabel="Number of Transactions"
)

Indeed, the middle of the week, especially Wednesday, is the time with the most volume of transactions. This indicates that information about the day of the week may encode relevant information for predicting future transactions. However, its usefulness depends on how the prediction target is defined. Since the current target aggregates purchases over multiple future days, the day of the week may provide limited additional predictive information compared to a target defined for a specific day.

#### 3.5.4 Hourly Activity

In [ ]:
hourly_events = (
    events
    .groupby(
        [
            events["timestamp"].dt.hour,
            "event"
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

hourly_events

In [ ]:
plot_line(
    df=hourly_events,
    x="timestamp",
    y="transaction",
    title="Hourly Activity",
    xlabel="Hour of The Day",
    ylabel="Number of Transactions"
)

There is also a difference in the volume of transactions across the hours of the day. However, given the aggregated nature of the target, this information will probably not be used as a feature

### 3.6 Conversion Funnel

This section analyzes how users progress through the purchase journey. Understanding where the largest drop-offs occur helps identify behavioral signals that may improve purchase prediction.

#### 3.6.1 Visitor Funnel

In [ ]:
visitor_funnel = build_funnel(visitor_events, "visitors")

visitor_funnel

In [ ]:
visitor_conversion = compute_conversion_rates(
    visitor_funnel,
    "visitors"
)

visitor_conversion

Only a small fraction of visitors progresses through every stage of the purchasing journey. The largest drop-off occurs before the add-to-cart event, suggesting that browsing behavior alone provides limited information about purchase intent.

#### 3.6.2 Item Funnel

In [ ]:
item_funnel = build_funnel(item_events, "items")

item_funnel

In [ ]:
item_conversion = compute_conversion_rates(
    item_funnel,
    "items"
)

item_conversion

Only a subset of products reaches the final purchase stage. This suggests that historical item popularity and conversion-related information may provide useful predictive signals.

### 3.7 Session Analysis

Purchasing decisions usually occur within browsing sessions rather than isolated events. This section estimates user sessions using a 30-minute inactivity threshold.

In [ ]:
SESSION_TIMEOUT = pd.Timedelta(minutes=30)

session_events = (
    events
    .sort_values(["visitorid", "timestamp"])
    .copy()
)

session_events["time_diff"] = (
    session_events
    .groupby("visitorid", sort=False)["timestamp"]
    .diff()
)

session_events["new_session"] = (
    session_events["time_diff"].isna()
    | session_events["time_diff"].gt(SESSION_TIMEOUT)
)

session_events["session_id"] = (
    session_events
    .groupby("visitorid", sort=False)["new_session"]
    .cumsum()
)

#### 3.7.1 Session Duration

In [ ]:
session_events["is_transaction"] = (
    session_events["event"] == "transaction"
).astype("int8")


sessions = (
    session_events
    .groupby(["visitorid", "session_id"], sort=False)
    .agg(
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        events=("event", "size"),
        transactions=("is_transaction", "sum"),
    )
    .reset_index()
)

sessions["duration_minutes"] = (
    sessions["end"]
    .sub(sessions["start"])
    .dt.total_seconds()
    .div(60)
)

sessions

In [ ]:
plot_histogram(
    sessions,
    "duration_minutes",
    bins=120,
    title="Session Duration",
    xlabel="Minutes",
    ylabel="Number of Sessions"
)

#### 3.7.2 Events per Session

In [ ]:
plot_histogram(
    sessions,
    "events",
    bins=120,
    title="Events per Session",
    xlabel="Number of Events",
    ylabel="Number of Sessions"
)

In [ ]:
sessions["has_transaction"] = sessions["transactions"] > 0

session_summary = conversion_summary(
    df=sessions,
    value_col="duration_minutes",
    converted_col="has_transaction",
    bins=[0, 1, 2, 5, 10, 20, 30, 60, 120, float("inf")],
    labels=["0-1", "1-2", "2-5", "5-10", "10-20", "20-30", "30-60", "60-120", ">120"],
    entity_name="sessions",
)

session_summary

Most sessions are short and contain relatively few interactions, while a small number of sessions are substantially longer and more active. Longer sessions have higher conversion rates, suggesting that session-based behavioral features may provide additional predictive information.

### 3.8 Recency Analysis

Recency is a common predictor of future user behavior. This section measures how recently visitors interacted with the platform.

In [ ]:
last_event = (
    events
    .groupby("visitorid")["timestamp"]
    .max()
)

reference_time = events["timestamp"].max()

recency = (
    (reference_time - last_event)
    .dt.total_seconds()
    / 3600
).rename("hours_since_last_event").reset_index()

recency

In [ ]:
plot_histogram(
    recency,
    "hours_since_last_event",
    bins=50,
    title="Hours Since Last Event",
    xlabel="Hours",
    ylabel="Number of Visitors"
)

In [ ]:
last_purchase = (
    events.loc[events["event"] == "transaction"]
    .groupby("visitorid")["timestamp"]
    .max()
)

purchase_recency = (
    (reference_time - last_purchase)
    .dt.total_seconds()
    / 3600
).rename("hours_since_last_purchase").reset_index()

purchase_recency

In [ ]:
plot_histogram(
    purchase_recency,
    "hours_since_last_purchase",
    bins=50,
    title="Hours Since Last Purchase",
    xlabel="Hours",
    ylabel="Number of Visitors"
)

In [ ]:
visitor_recency = recency.merge(
    visitor_events[["visitorid", "transaction"]],
    on="visitorid",
)

visitor_recency["has_transaction"] = (
    visitor_recency["transaction"] > 0
)

In [ ]:
recency_summary = conversion_summary(
    df=visitor_recency,
    value_col="hours_since_last_event",
    converted_col="has_transaction",
    bins=[0, 1, 6, 12, 24, 48, 72, 24*7, 24*14, 24*30, float("inf")],
    labels=["<1h", "1-6h", "6-12h", "12-24h", "1-2d", "2-3d", "3-7d", "1-2w", "2-4w", ">30d",],
    entity_name="visitors",
    converted_name="purchasers",
)

recency_summary

Recency alone does not strongly differentiate visitors who have purchased at least once from those who have not. Since this binary distinction may hide differences in purchasing intensity, we further investigate whether visitors with more recent activity also tend to accumulate a higher number of transactions.

In [ ]:
recency_transactions = (
    visitor_recency
    .assign(
        bin=pd.cut(
            visitor_recency["hours_since_last_event"],
            bins=[0, 1, 6, 12, 24, 48, 72, 24*7, 24*14, 24*30, float("inf")],
            labels=["<1h", "1-6h", "6-12h", "12-24h", "1-2d", "2-3d", "3-7d", "1-2w", "2-4w", ">30d"],
            include_lowest=True,
        )
    )
    .groupby("bin", observed=True)
    .agg(
        visitors=("visitorid", "count"),
        transactions=("transaction", "sum"),
    )
)

recency_transactions["transactions_per_visitor"] = (
    recency_transactions["transactions"]
    / recency_transactions["visitors"]
)

recency_transactions

Visitors with the most recent activity exhibit the highest average number of transactions per visitor. However, this relationship is neither strong nor consistently monotonic across all recency intervals. Because this analysis is based on aggregated historical behavior rather than prediction snapshots, it is insufficient to determine the predictive value of recency. The contribution of recency-based features will therefore be evaluated during model training, and these features may remain secondary if other behavioral signals provide stronger predictive information.

## 4. Item Properties

The item properties dataset contains historical metadata describing product characteristics over time. Unlike the events dataset, which captures user interactions, this dataset provides contextual information that may enrich behavioral features during feature engineering.

### 4.1 Overview

This section provides a general overview of the available item properties.

In [ ]:
print(f"Memory usage: {item_properties.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

item_properties

item_properties.info()

item_properties.describe(include="all")

### 4.2 Property Distribution

Different properties occur with very different frequencies. Understanding which properties are more common helps identify candidates for feature engineering.

In [ ]:
property_counts = (
    item_properties["property"]
    .value_counts()
    .rename("count")
    .reset_index()
    .rename(columns={"index": "property"})
)

property_counts.head(30)

In [ ]:
top30_properties = property_counts.head(30)

plot_bar(
    df=top30_properties,
    x="property",
    y="count",
    title="Top 30 Most Frequent Properties",
    xlabel="Property",
    ylabel="Occurrences",
    order=top30_properties["property"],
    rotation=90,
)

Only a subset of properties appears frequently throughout the catalog. These properties represent potential candidates for feature engineering, but incorporating item-level metadata would increase the complexity of the feature pipeline. Therefore, property-based features are considered a secondary source of information and can be explored if behavioral and item-popularity features provide insufficient predictive performance.

### 4.3 Property History

Item properties are versioned over time, meaning that the same product may receive multiple property updates throughout its lifecycle.

In [ ]:
property_history = (
    item_properties
    .groupby("itemid")
    .size()
    .rename("records")
    .reset_index()
)

property_history.describe()

In [ ]:
plot_histogram(
    property_history,
    column="records",
    bins=100,
    title="Property Records per Item",
    xlabel="Number of Property Records",
    ylabel="Number of Items",
)

Many products have multiple property records, indicating that item metadata changes over time. During feature engineering, only the property values available before each prediction snapshot should be used, preventing target leakage.

Because incorporating this historical metadata requires additional temporal processing, property-history features are treated as secondary candidates and may be introduced if the primary behavioral and popularity-based features are not sufficient.

## 5. Category Tree

The category tree defines the hierarchical organization of product categories and enables aggregation at multiple levels.

### 5.1 Overview

In [ ]:
print(f"Memory usage: {category_tree.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

category_tree

category_tree.info()

### 5.2 Root and Child Categories

During data cleaning, missing parentid values were replaced with -1 to explicitly mark root categories.

In [ ]:
category_tree.describe(include="all")

In [ ]:
root_categories = (category_tree["parentid"] == -1).sum()
child_categories = len(category_tree) - root_categories

print(f"Root categories: {root_categories}")
print(f"Child categories: {child_categories}")
category_tree["parentid"].value_counts().head(20)

The `categoryid` and `parentid` fields define parent-child relationships between categories. Categories with `parentid = -1` represent the roots of this structure, while other categories reference another category as their parent.

This structure can be explored as a hierarchy by following the parent references and calculating the depth of each category.

### 5.3 Hierarchy Depth

To understand the structure of the category relationships, we calculate the depth of each category by recursively following its parent references until reaching a root category.

The resulting depth distribution shows how many categories exist at each level of the hierarchy.

In [ ]:
category_depth = build_category_depth(category_tree)

category_depth["depth"].value_counts().sort_index()
plot_bar(
    df=category_depth["depth"].value_counts().sort_index().reset_index(),
    x="depth",
    y="count",
    title="Category Depth Distribution",
    xlabel="Depth",
    ylabel="Number of Categories",
)

The depth distribution can help determine whether category-level features should be represented at a broad level, such as the root category, or at more specific levels of the hierarchy.

### 5.4 Category-Level Purchase Behavior

Category depth alone does not determine which hierarchy level should be used for feature engineering. To evaluate the usefulness of category information, purchase behavior is compared across the available category levels.

Each item may be associated with one or more categories. These associations are expanded so that each item-category relationship can be analyzed separately.


In [ ]:
item_categories = (
    item_properties.loc[
        item_properties["property"] == "categoryid",
        ["itemid", "value"]
    ]
    .dropna()
    .assign(
        categoryid=lambda df: df["value"].str.split()
    )
    .explode("categoryid")
    .drop(columns="value")
)

item_categories.head(10)

The resulting dataset associates each item with its corresponding category or categories. This allows event-level interactions to be analyzed according to product category.

In [ ]:
category_events = (
    events[["timestamp", "visitorid", "event", "itemid"]]
    .merge(
        item_categories,
        on="itemid",
        how="inner"
    )
)

category_events.head()

The number of items associated with each category varies considerably, indicating that some categories are substantially more represented in the catalog than others.

In [ ]:
category_items = (
    item_categories
    .groupby("categoryid")["itemid"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="items")
)

category_items.describe()

In [ ]:
top_categories = category_items.head(30)

plot_bar(
    df=top_categories,
    x="categoryid",
    y="items",
    title="Top 30 Categories by Number of Items",
    xlabel="Category ID",
    ylabel="Number of Items",
    rotation=90,
    order=top_categories["categoryid"]
)

Purchase behavior also varies substantially across categories. Some categories have considerably higher conversion rates than others, even when considering only categories with a minimum number of views.

In [ ]:
category_purchases = (
    category_events
    .groupby("categoryid")
    .agg(
        items=("itemid", "nunique"),
        views=("event", lambda x: (x == "view").sum()),
        transactions=("event", lambda x: (x == "transaction").sum()),
    )
    .reset_index()
)

category_purchases["conversion_rate"] = (
    category_purchases["transactions"]
    / category_purchases["views"]
    * 100
)

category_purchases.sort_values(
    "transactions",
    ascending=False
).head(20)

In [ ]:
category_purchases[
    category_purchases["views"] >= 100
].sort_values(
    "conversion_rate",
    ascending=False
).head(20)

The differences in conversion rates suggest that category information may contain useful predictive information. However, incorporating category-level historical features would require additional temporal aggregation and category-level processing.

Therefore, category-based features are considered secondary candidates for the Gold layer. The initial feature engineering effort prioritizes user behavioral signals and item-popularity information, while category-based features can be incorporated later if these signals are insufficient.

To further investigate the category hierarchy, the analysis also examines purchase behavior at the parent-category level. This helps determine whether aggregating categories into broader groups produces similar or different purchasing patterns.

In [ ]:
category_levels = category_tree[
    ["categoryid", "parentid"]
].copy()

category_levels["categoryid"] = (
    category_levels["categoryid"].astype(str)
)

category_levels["parentid"] = (
    category_levels["parentid"].astype(str)
)

In [ ]:
category_purchases = (
    category_events
    .groupby("categoryid")
    .agg(
        items=("itemid", "nunique"),
        views=("event", lambda x: (x == "view").sum()),
        transactions=("event", lambda x: (x == "transaction").sum()),
    )
    .reset_index()
)

category_purchases["conversion_rate"] = (
    category_purchases["transactions"]
    / category_purchases["views"]
    * 100
)

category_purchases

In [ ]:
category_purchases = category_purchases.copy()

category_purchases["categoryid"] = (
    category_purchases["categoryid"]
    .astype(str)
    .str.replace(r"^\['", "", regex=True)
    .str.replace(r"'\]$", "", regex=True)
)

category_tree["categoryid"] = category_tree["categoryid"].astype(str)
category_tree["parentid"] = category_tree["parentid"].astype(str)

category_purchases = category_purchases.merge(
    category_tree[["categoryid", "parentid"]],
    on="categoryid",
    how="left",
    validate="many_to_one"
)

In [ ]:
parent_purchases = (
    category_purchases
    .groupby("parentid")
    .agg(
        categories=("categoryid", "nunique"),
        items=("items", "sum"),
        views=("views", "sum"),
        transactions=("transactions", "sum"),
    )
    .reset_index()
)

parent_purchases["conversion_rate"] = (
    parent_purchases["transactions"] / parent_purchases["views"] * 100
)

parent_purchases.sort_values("conversion_rate", ascending=False).head(20)

Aggregating the observed purchase behavior by parent category also produces differences in conversion rates. However, this analysis alone does not establish that the parent level is more informative than the specific category level. In addition, the category hierarchy contains multiple depths, so the direct parent is not necessarily equivalent to the root category.

Therefore, the EDA establishes that category information is relevant enough to be considered as a potential source of predictive information, but it does not conclusively determine the optimal hierarchy level. Given the additional complexity required to incorporate this information, category hierarchy features are kept as secondary candidates for future feature engineering.

## 6. Candidate Features

The exploratory analysis identified several behavioral, temporal, and item-level patterns that may contribute to predicting future purchases.

Based on these findings, the initial feature engineering effort prioritizes recent user behavior, historical purchasing activity, session engagement, and item popularity. These signals provide direct representations of visitor intent and product-level demand while keeping the feature pipeline relatively manageable.

Category and item-property information are also considered potential sources of predictive information, but are treated as secondary candidates due to the additional complexity required to incorporate their historical and hierarchical structure.

The final feature selection will be driven by model performance and predictive importance during training.

### 6.1 Selected Features

| Feature |	Description |	Motivation |
|--------|----------|--------|
| `user_views_7d` |	Number of product views by the visitor during the last 7 days |	Captures recent browsing activity.|
| `user_addtocarts_7d`	|Number of add-to-cart events during the last 7 days|	Stronger purchase intent than views.|
| `user_transactions_30d`|	Number of previous purchases in the last 30 days|	Historical purchasing behavior was highly concentrated among a small subset of users.|
| `user_sessions_30d` |Number of sessions during the recent period	|Represents overall engagement.|
| `avg_session_duration_30d` |Average duration of recent sessions	|Longer sessions exhibited higher conversion rates.|
| `user_cart_items_avg_popularity_7d` | Average historical popularity of items added to the visitor's cart during the last 7 days | Captures whether the visitor is interacting with items that historically receive higher purchase volume.|
| `user_cart_top_seller_ratio_7d` | Proportion of items added to the visitor's cart that belong to the historically most popular items | Captures whether the visitor's cart activity is concentrated on highly popular products.|

### 6.2 Secondary Feature Candidates

Several additional sources of information were identified during the exploratory analysis but are not prioritized in the initial feature set.

| Information source | Potential use |
|---|---|
| Category hierarchy | Represent user interactions and purchasing behavior at different category levels. |
| Item properties | Capture product characteristics and their historical changes. |
| Recency-based features | Represent how recently a visitor interacted with the platform. |
| Hour-of-day features | Capture intraday temporal purchasing patterns. |

These candidates remain available for future iterations. They can be incorporated if the primary behavioral and item-popularity features provide insufficient predictive performance or if additional business requirements justify the added complexity.

### 6.3 Excluded Features

The following information will not be used directly as model features.

| Feature | Reason |
|----------|--------|
| `visitorid` | High-cardinality identifier that does not generalize to unseen visitors. |
| `itemid` | High-cardinality identifier that encourages memorization rather than learning general behavioral patterns. Aggregate item-popularity features are used instead. |
| Raw event history | Historical interactions will be summarized into aggregated behavioral features. |